In [1]:
import torch
from torch import Tensor
from nba_api.stats.static import teams
from pandas import DataFrame
from typing import List, Tuple

In [2]:
import requests

In [3]:
all_teams = teams.get_teams()

In [4]:
team = [team for team in all_teams if team['abbreviation'] == 'ATL'][0]
team_id = team['id']
print(team_id)

1610612737


In [6]:
from nba_api.stats.endpoints import leaguegamefinder
custom_headers = {
    'Host': 'stats.nba.com',
    'Connection': 'keep-alive',
    'Cache-Control': 'max-age=0',
    'Upgrade-Insecure-Requests': '1',
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_3) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/73.0.3683.86 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3',
    'Accept-Encoding': 'gzip, deflate, br',
    'Accept-Language': 'en-US,en;q=0.9',
    'x-nba-stats-origin': 'stats',
    'x-nba-stats-token': 'true'
}
gamefinder = leaguegamefinder.LeagueGameFinder(team_id_nullable=team_id, headers=custom_headers, timeout=120)
games = gamefinder.get_data_frames()
print(games[0].head())

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=120)

In [ ]:
# takes in the id of team A and the abbreviation of team B and returns their 5 latest matchups
def last_five_games_against_b(a_id: str, b_abr: str):
    gamefinder = leaguegamefinder.LeagueGameFinder(team_id_nullable=a_id)
    a_games = gamefinder.get_data_frames()[0]
    return a_games[a_games.MATCHUP.str.contains(b_abr)].head()


In [ ]:
matchups = last_five_games_against_b(team_id, 'CHA')
matchups

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [6]:
def season_from_id(season_id: str):
    start_year = season_id[-2:]
    end_year = str(int(start_year)+1)
    return f'20{start_year}-{end_year}'

In [7]:
# wrapper for empty tensor
if torch.cuda.is_available():
    print("CUDA is available. Using GPU.")
    device = torch.device("cuda")
else:
    print("CUDA not available. Using CPU.")
    device = torch.device("cpu")

def T(shape: tuple) -> torch.Tensor:
    return torch.empty(shape, device=device, dtype=torch.float32)

CUDA is available. Using GPU.


In [8]:
# wrapper for random tensor
def Tr(shape: tuple) -> torch.Tensor:
    return torch.rand(shape, device=device, dtype=torch.float32)

In [9]:
# quantities included in input tensor
num_quantities = 20

In [10]:
from nba_api.stats.endpoints import boxscoretraditionalv3
from nba_api.stats.endpoints import commonteamroster
import numpy as np

# get box scores of each player for each game in df of games of a single team
def get_box_scores(games: DataFrame, num_quantities: int, 
                   device: torch.device | None = None) -> Tuple[List[List[DataFrame]], str, str, str]:
    if not device:
        if torch.cuda.is_available():
            print("CUDA is available. Using GPU.")
            device = torch.device("cuda")
        else:
            print("CUDA not available. Using CPU.")
            device = torch.device("cpu")

    
    team_id = games['TEAM_ID'].iloc[0]
    season = season_from_id(games['SEASON_ID'].iloc[0]) # most recent

    a_roster = (
        commonteamroster.CommonTeamRoster(team_id=team_id, season=season).
        get_data_frames()[0].
        PLAYER_ID
    )

    a_roster_map = {player_id: i for i, player_id in enumerate(a_roster)}

    input = T(((len(games)), len(a_roster), num_quantities))

    for i, game in enumerate(games.itertuples()):

        game_id = game.GAME_ID
        opp_abv = game.MATCHUP[-3:]
        opp_id = teams.find_team_by_abbreviation(opp_abv)['id']

        # filter box scores by players in current roster and quantitative metrics
        box_scores = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=game_id).get_data_frames()[0]
        box_scores = box_scores[box_scores["personId"].isin(a_roster_map)]

        # reformat minutes str -> float
        time = box_scores['minutes'].str.split(":")
        mins = time.str[0].replace("", "0").astype(int)
        secs = time.str[1].fillna(0).replace("", "0").astype(int)
        box_scores['minutes'] = mins + secs / 60
        
        box_scores_quant = box_scores.iloc[:,14:]
        print(box_scores.columns)
        
        # sort box scores into input tensor using personId -> index mapping
        rows = box_scores['personId'].map(a_roster_map).to_numpy()
        input[i][rows] = torch.from_numpy(box_scores_quant.to_numpy()).to(device).float()

    return input, len(a_roster_map)
        

In [11]:
def last_t_games(id, t):
    gamefinder = leaguegamefinder.LeagueGameFinder(team_id_nullable=id)
    games = gamefinder.get_data_frames()[0]
    return games.head(t)

In [12]:
t = 10
games = last_t_games(team_id, t)
box_scores, num_players = get_box_scores(games, num_quantities, device)

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [ ]:
DataFrame(Tensor.cpu(box_scores[0]))

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.000000,0.0,0.0,0.000,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.0,0.0,0.000,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,34.450001,7.0,15.0,0.467,1.0,4.0,0.250,4.0,5.0,0.80,1.0,12.0,13.0,9.0,4.0,1.0,3.0,1.0,19.0,-5.0
3,0.000000,0.0,0.0,0.000,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31.966667,6.0,17.0,0.353,1.0,8.0,0.125,4.0,5.0,0.80,2.0,4.0,6.0,8.0,0.0,1.0,3.0,2.0,17.0,7.0
5,0.000000,0.0,0.0,0.000,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,25.483334,9.0,12.0,0.750,0.0,1.0,0.000,3.0,4.0,0.75,1.0,2.0,3.0,4.0,0.0,0.0,2.0,2.0,21.0,-10.0
7,40.250000,2.0,13.0,0.154,1.0,7.0,0.143,5.0,5.0,1.00,0.0,3.0,3.0,4.0,1.0,2.0,3.0,4.0,10.0,8.0
8,0.000000,0.0,0.0,0.000,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.000000,0.0,0.0,0.000,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
from torch.nn import LSTM

# forecasting performance of team A
torch.manual_seed(4188)
a_rnn = LSTM(input_size=num_quantities, hidden_size=num_quantities).to(device)
a_h0 = Tr((1,num_players,num_quantities))
a_c0 = Tr((1,num_players,num_quantities))
output, (a_hn, a_cn) = a_rnn(box_scores, (a_h0, a_c0))

In [ ]:
def get_season_from_game_id(game_id: str) -> str:
    start_y = game_id[3:5]
    return f'20{start_y}-{str(int(start_y)+1)}'

In [ ]:
def reformat_box_score(box_score: DataFrame) -> DataFrame:
    print(box_score['minutes'].replace('','0:0').str.split(':'))
    min = box_score['minutes'][0].replace('', '0').split(':')[0]
    sec = box_score['minutes'][1].replace('', '0').split(':')[1]
    print(min)
    box_score['minutes'] = int(min) + int(sec) / 60
    return box_score

In [ ]:
def get_box_score_from_game_id(game_id: str) -> DataFrame:
    box_score =  boxscoretraditionalv3.BoxScoreTraditionalV3(game_id=game_id).get_data_frames()[0]
    reformatted = reformat_box_score(box_score)
    return reformatted

In [ ]:
from pandas import Series

# converts pandas structure to tensor, pandas -> np -> tensor
def Tp(pandas: DataFrame | Series):
    print(type(pandas.to_numpy()))
    return torch.from_numpy(pandas.to_numpy())

In [ ]:
from nba_api.stats.endpoints import playergamelog
import pandas as pd

# get a player's last T box scores predating a given box scores dataframe
def get_t_box_scores_prior_to_game(pid: str, game: DataFrame, t: int):
    game_id = game.gameId.iloc[0]
    game_log = playergamelog.PlayerGameLog(
        player_id=pid,
        season=get_season_from_game_id(game_id),
        season_type_all_star="Regular Season"
    )
    df = game_log.get_data_frames()[0]
    df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
    df = df.sort_values("GAME_DATE").reset_index(drop=True)

    idx = df.index[df['Game_ID'] == game_id][0]
    last_t_game_ids = df.iloc[max(0, idx-t+1):idx+1]['Game_ID'].reset_index(drop=True)

    box_scores = Tp(last_t_game_ids.map(get_box_score_from_game_id))
    print(type(box_scores))
    return last_t_game_ids

In [ ]:
box_scores = boxscoretraditionalv3.BoxScoreTraditionalV3(game_id='0022500775').get_data_frames()[0]
get_t_box_scores_prior_to_game(1642258, box_scores, 5)

0     [21, 58]
1     [33, 39]
2     [30, 13]
3     [35, 45]
4     [28, 04]
5     [27, 32]
6     [17, 47]
7     [18, 33]
8     [16, 07]
9      [7, 36]
10     [1, 23]
11     [1, 23]
12      [0, 0]
13      [0, 0]
14      [0, 0]
15    [22, 55]
16    [36, 17]
17    [11, 05]
18    [21, 34]
19    [31, 23]
20    [27, 36]
21    [26, 24]
22    [21, 39]
23    [16, 39]
24    [23, 05]
25     [1, 23]
26      [0, 0]
Name: minutes, dtype: object
02010
0     [19, 15]
1     [38, 39]
2     [31, 31]
3     [34, 35]
4     [33, 42]
5     [31, 53]
6     [24, 34]
7     [16, 29]
8      [9, 23]
9       [0, 0]
10      [0, 0]
11      [0, 0]
12    [37, 36]
13    [23, 26]
14    [28, 25]
15    [40, 21]
16    [48, 00]
17    [29, 34]
18    [22, 42]
19     [9, 56]
20      [0, 0]
21      [0, 0]
22      [0, 0]
23      [0, 0]
Name: minutes, dtype: object
01090
0     [23, 29]
1     [39, 54]
2     [30, 51]
3     [31, 06]
4     [37, 37]
5     [28, 20]
6     [23, 28]
7     [17, 09]
8      [8, 06]
9       [0, 0]
10      [0, 0]


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.